# Travdata från ATG – inspektion och datatvätt

## 1. Kontext

Datasetet innehåller startlistor och resultat från svenska travlopp hämtade via ATG:s öppna API. Varje rad representerar **en hästs start i ett lopp** — inte ett lopp i sig.

**Varifrån kommer datan?**  
API:t `atg.se/services/racinginfo/v1/api` exponerar historiska resultat för spelformerna V64, V85 och V86. Datan hämtades och normaliserades i `01_setup/`.

**Vilken population representerar den?**  
Enbart hästar i ATG:s kupongspel — redan ett urval av de starkaste hästarna i sin kategori. Slutsatser gäller inte travhästar i allmänhet.

**Vad begränsar analysen?**  
- Kort tidsperiod (feb–maj 2026) — säsongseffekter syns inte  
- Inga tränings-, konditions- eller veterinärdata ingår

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

PROCESSED_DIR = Path.cwd().parent.parent / "data" / "processed"
df = pd.read_csv(PROCESSED_DIR / "starters.csv", parse_dates=["date"])

print(f"Rader: {df.shape[0]:,}  |  Kolumner: {df.shape[1]}")
df.info()

# Grundläggande statistik


In [133]:
df.describe()

,date,division,distance_m,start_number,post_position,horse_id,horse_age,driver_id,trainer_id,finish_position,odds,race_time_sec,prize_money,won
count,6527,6527.000000,6527.000000,6527.000000,6297.000000,6159.000000,6527.000000,6121.000000,6133.000000,6089.000000,6141.000000,5298.000000,6.022000e+03,6527.000000
mean,2026-03-29 13:36:18.091006,7.186916,2074.957867,6.205761,5.412895,782771.467608,5.992033,417335.434896,352912.166803,8.146494,37.356113,159.311136,1.740196e+04,0.090087
min,2026-02-09 00:00:00,2.000000,1200.000000,1.000000,1.000000,722237.000000,3.000000,317.000000,334.000000,1.000000,1.080000,112.147300,1.000000e+03,0.000000
25%,2026-03-06 00:00:00,6.000000,2100.000000,3.000000,3.000000,774255.000000,5.000000,125464.000000,104149.000000,3.000000,6.630000,154.936000,2.291000e+03,0.000000
50%,2026-04-01 00:00:00,7.000000,2140.000000,6.000000,5.000000,785019.000000,6.000000,545161.000000,191875.000000,6.000000,18.170000,159.858000,6.400000e+03,0.000000
75%,2026-04-22 00:00:00,9.000000,2140.000000,9.000000,8.000000,792262.000000,7.000000,699497.000000,591841.000000,9.000000,53.060000,164.566000,1.750000e+04,0.000000
max,2026-05-14 00:00:00,12.000000,3140.000000,15.000000,15.000000,815238.000000,15.000000,859144.000000,858645.000000,45.000000,360.770000,249.216000,1.500000e+06,1.000000
std,NaN,2.036819,303.445200,3.483569,3.266841,14633.635224,1.956608,290140.630475,265158.861050,9.722001,44.924706,24.288882,4.704443e+04,0.286329


# Fördelning per speltyp


In [135]:
print(df["game_type"].value_counts().to_string())

game_type
V64    3774
V85    1553
V86    1200


# Fördelning per distans


In [136]:
print(df["distance_m"].value_counts().sort_index().to_string())

distance_m
1200      68
1400       9
1600      70
1609     120
1640    1096
1700      43
1720      24
1730      16
1800      12
2100     329
2140    4077
2200      41
2300      26
2350       9
2400       5
2600      27
2640     451
2900      14
3140      90


## Observation: Spårfördel på 2140m

- Spår 2 och 6 har högst vinstprocent på 2140m med 11.5% respektive 11.9%.
- Spår 8 och uppåt försämras tydligt – ytterlägen är en nackdel.

In [137]:
print(subset.groupby("post_position")["won"].agg(["count", "sum", "mean"]).to_string())


               count  sum      mean
post_position                      
1.0              484   43  0.088843
2.0              452   52  0.115044
3.0              454   46  0.101322
4.0              428   44  0.102804
5.0              367   35  0.095368
6.0              362   43  0.118785
7.0              313   29  0.092652
8.0              227   13  0.057269
9.0              261   20  0.076628
10.0             215   22  0.102326
11.0             177    8  0.045198
12.0             104    2  0.019231
13.0               9    0  0.000000
14.0               9    2  0.222222
15.0               8    0  0.000000


# Strukna hästar

Total rows:    6,527
Scratched:     333 (5.1%)

Mycket relevant. 

In [138]:
print(f"Total rows:    {len(df):,}")
print(f"Scratched:     {df['scratched'].sum():,} ({df['scratched'].mean()*100:.1f}%)")

Total rows:    6,527
Scratched:     333 (5.1%)


## NaN per kolumn

- `race_name` – 1402 saknar namn, förväntat. Vanliga lopp har inget namn, endast speciallopp som t.ex. Elitloppet.
- `finish_position`, `odds`, `prize_money` – matchar antal strukna hästar, förväntat.
- `race_time_sec` – ATG rapporterar inte alltid tid, inget vi kan åtgärda.
- `start_method`, `post_position` – samma 230 rader, troligen samma lopp saknar denna info.
- `horse_id`, `driver_id`, `trainer_id` – ATG returnerar 0 för okända, ersatte med NaN.


In [139]:
null_counts = df.isnull().sum()
print(null_counts[null_counts > 0].to_string())

race_name          1402
start_method        230
post_position       230
horse_id            368
driver_id           406
trainer_id          394
finish_position     438
odds                386
race_time_sec      1229
prize_money         505


# Fördelning per bana


In [140]:
print(df["track"].value_counts().to_string())

track
Solvalla           694
Åby                650
Jägersro           479
Axevalla           418
Bergsåker          407
Halmstad           401
Gävle              372
Bollnäs            370
Färjestad          367
Romme              363
Umåker             298
Boden              280
Örebro             234
Bjerke             231
Mantorp            205
Eskilstuna         192
Klosterskogen      147
Jägersro Galopp     90
Bro Park            87
Rättvik             68
Orkla               62
Östersund           59
Göteborg Galopp     53


## 3. Datatvätt

Tre medvetna beslut — varje steg motiveras innan det utförs.

In [ ]:
# Beslut 1: Filtrera bort strukna starter
# Motivering: en häst som aldrig startade har inget jämförbart utfall.
# Att inkludera dem i vinstfrekvensberäkningar skulle ge missvisande siffror.
active = df[~df["scratched"]].copy()
print(f"Borttagna (scratched): {df['scratched'].sum():,} rader ({df['scratched'].mean():.1%})")

# Beslut 2: Normalisera startmetod med numpy where
# Motivering: API:t returnerar varianter som 'Voltstart', 'volt', 'Auto' — samma koncept.
# Kategoriserar till volt/auto för att kunna jämföra startmetoder.
active["startmetod"] = np.where(
    active["start_method"].str.lower().str.contains("volt", na=False), "Volt", "Auto"
)

# Beslut 3: Klipp orimliga loptider
# Motivering: loptider under 60 eller över 200 sek är sannolikt datainmatningsfel.
# Påverkar hastighetsjämförelser om de lämnas kvar.
before = len(active)
active = active[active["race_time_sec"].isna() | active["race_time_sec"].between(60, 200)]
print(f"Borttagna (orimlig loptid): {before - len(active)} rader")
print(f"Rader kvar för analys: {len(active):,}")

# Subset för 2140m-analys i nästa cell
subset = active[active["distance_m"] == 2140]

## 4. Vad inspektionen visade

**Mönster som syns i tabellerna:**
- 2140m dominerar datasetet (62% av alla starter) — analyser per distans bör ta hänsyn till detta skeva urval.
- Spår 8+ på 2140m har tydligt lägre vinstfrekvens. Ytterlägen är en strukturell nackdel.
- V64 svarar för 58% av starterna — resultat viktade av speltyp kan skilja sig.

**Vad datan inte kan berätta:**
- Varför vissa lopp saknar `start_method` och `post_position` (230 rader) — troligen ett API-fel vid insamlingen, inte ett travfenomen.
- Om resultaten är representativa för hela säsongen — perioden feb–maj 2026 är för kort för att fånga säsongsvariation.